# Audiobook Engine: Stage 1, PDF reconnaissance

CPU-only preparation for *Buddhist Yoga: A Comprehensive Course*. Extract and audit the manuscript before using GPU time. No OCR, silent rewriting, or audio synthesis occurs in this notebook. Results are saved to Google Drive.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, tarfile

REPOSITORY = 'https://github.com/privacyfreak84/colabchatterLLMthingstuff.git'
REPO_REF = 'feat/manuscript-repair-review'
PROJECT_DIR = Path('/content/colabchatterLLMthingstuff')
ENGINE_DIR = PROJECT_DIR / 'engine'
PDF_PATH = PROJECT_DIR / 'Buddhist_Yoga_TTS_clean.pdf'
ASSET_DIR = PROJECT_DIR / 'assets'
ARCHIVE_PATH = PROJECT_DIR / 'audiobook-engine-starter.tar.xz'
EXPECTED_ARCHIVE_BYTES = 22116
EXPECTED_ARCHIVE_SHA256 = '69ae62a5ec94d2784b910595ddf642a8acdc02d19029c5660dc3c66026615463'

if PROJECT_DIR.exists() and (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPOSITORY, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(f'Project path exists but is not a Git checkout: {PROJECT_DIR}')
assert PDF_PATH.is_file(), f'Source PDF not found: {PDF_PATH}'
parts = [ASSET_DIR / f'audiobook-engine-starter.tar.xz.part-{i:02d}' for i in range(6)]
missing = [str(p) for p in parts if not p.is_file()]
assert not missing, f'Engine bundle segments missing: {missing}'
if not ARCHIVE_PATH.is_file():
    ARCHIVE_PATH.write_bytes(b''.join(p.read_bytes() for p in parts))
archive_bytes = ARCHIVE_PATH.read_bytes()
assert len(archive_bytes) == EXPECTED_ARCHIVE_BYTES, f'Unexpected bundle size: {len(archive_bytes)}'
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
assert archive_sha256 == EXPECTED_ARCHIVE_SHA256, f'Bundle checksum mismatch: {archive_sha256}'
if not (ENGINE_DIR / 'audiobook' / 'pdf_ingest.py').is_file():
    ENGINE_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVE_PATH, mode='r:xz') as bundle:
        bundle.extractall(ENGINE_DIR)
sys.path.insert(0, str(ENGINE_DIR))
print('Project:', PROJECT_DIR)
print('Engine:', ENGINE_DIR)
print('PDF:', PDF_PATH, f'({PDF_PATH.stat().st_size:,} bytes)')
print('Engine bundle checksum:', archive_sha256)


## 1. Install document dependencies

Only PyMuPDF and PyYAML are installed here. Colab's preinstalled CUDA-enabled PyTorch is left untouched. Chatterbox comes later, after the text is reviewed.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ENGINE_DIR / 'requirements-colab.txt')], check=True)
print('Document dependencies ready.')


## 2. Extract, validate, and export the manuscript

The extraction keeps page/block provenance. It creates separate exports for the preface, eight chapters, and notes, plus a pronunciation-candidate review file. No candidate respelling is automatically applied.

In [ ]:
from audiobook.pdf_ingest import extract_pdf, build_report, write_extraction
from audiobook.validation import validate_project_document
from audiobook.structure import export_sections
from audiobook.linguistic_analysis import export_pronunciation_inventory
import yaml

config_path = ENGINE_DIR / 'config' / 'buddhist_yoga.yaml'
with config_path.open('r', encoding='utf-8') as f:
    project_config = yaml.safe_load(f) or {}
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/Audiobook/Buddhist Yoga/reconnaissance')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

result = extract_pdf(PDF_PATH)
expected_chapters = len(project_config.get('chapters', []))
paths = write_extraction(result, OUTPUT_DIR, expected_chapters=expected_chapters)
report = build_report(result, expected_chapters=expected_chapters)
validation = validate_project_document(result, project_config)
section_paths = export_sections(result, project_config, OUTPUT_DIR)
pronunciation_paths = export_pronunciation_inventory(result, project_config, OUTPUT_DIR)
(OUTPUT_DIR / 'project_validation.json').write_text(json.dumps(validation, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps({'pages': result.page_count, 'words': result.word_count, 'characters': result.char_count, 'outline_entries': len(result.outline), 'findings': result.findings, 'chapter_coverage': report.get('chapter_coverage'), 'project_validation_passed': validation['passed'], 'outputs': {k: str(v) for k, v in paths.items()}, 'section_outputs': {k: str(v) for k, v in section_paths.items()}, 'pronunciation_outputs': {k: str(v) for k, v in pronunciation_paths.items()}}, indent=2))


## 3. Inspect document structure and extraction warnings

In [ ]:
for entry in result.outline:
    print(f"{'  ' * max(0, entry.level - 1)}p.{entry.page:>3}  {entry.title}")
print('\nFindings:')
for finding in result.findings:
    print('-', finding)


## 4. Read samples before approving the manuscript

The plain-text manuscript is convenient for reading. The JSONL block file retains page and bounding-box provenance. Review the opening, ending, chapter boundaries, quotations, and suspicious hyphenation before continuing.

In [ ]:
manuscript = paths['text'].read_text(encoding='utf-8')
print(manuscript[:5000])
print('\n--- END SAMPLE ---')
print(manuscript[-2500:])


## 5. Build a reversible narration-repair proposal

This pass uses page/block provenance to identify likely sentence continuations. Cross-page joins are applied only to separate *proposed* copies when an exact adjacent-block match is unique. Same-page candidates are listed for manual review only. The source exports are never overwritten. It also prepares a focused pronunciation dictionary draft; all spoken forms remain blank pending auditions and approval.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR / 'tools'))
from manuscript_repair import build_review_bundle

repair_summary = build_review_bundle(
    blocks_jsonl=paths['blocks'],
    section_manifest=OUTPUT_DIR / 'section_manifest.json',
    main_manuscript=OUTPUT_DIR / 'main_manuscript.txt',
    notes_manuscript=OUTPUT_DIR / 'notes_manuscript.txt',
    pronunciation_candidates=OUTPUT_DIR / 'pronunciation_candidates.json',
    output_dir=OUTPUT_DIR,
)
print(json.dumps(repair_summary, ensure_ascii=False, indent=2))


## Stop here

Do not launch synthesis yet. Review the original and proposed main/Notes manuscripts, every row in `manuscript_repair_review.csv`, the optional title-page opening, and `pronunciation_dictionary_draft.csv`. Proposed joins are not approved speech text. The next stage begins only after the user approves the manuscript and pronunciation choices.